**IMPORTS**

In [24]:
from Configuracion import CARGADOS, PROCESADOS, CSV_GENERADOS, IMPLEMENTACION_GOBERNANZA, ANALITICOS_PARQUET, ANALITICOS_RESULTADOS
from Configuracion import RECURSOS

import os
import re
from collections import Counter
from pyspark.sql.types import IntegerType

**SESION DE SPARK**

In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
        
    .appName("CybersecurityDataAnalytics")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "10")
    .getOrCreate()
)

**RUTAS**

In [26]:
path_crudo = os.path.join(PROCESADOS, "dataset_crudo_unificado.parquet")
path_ml    = os.path.join(PROCESADOS, "dataset_ml_unificado.parquet")

os.makedirs(ANALITICOS_PARQUET, exist_ok=True)

**CARGAR LOS DATOS**

In [27]:
df_crudo = spark.read.parquet(path_crudo)
df_ml    = spark.read.parquet(path_ml)

print("df_crudo filas:", df_crudo.count(), "| columnas:", len(df_crudo.columns))
print("df_ml filas:", df_ml.count(), "| columnas:", len(df_ml.columns))

df_crudo filas: 3119345 | columnas: 85
df_ml filas: 2830743 | columnas: 79


**NORMALIZACION DE NOMBRES Y DUPLICADOS (ANOMALIA DE COLUMNA DUPLICADA DEL DATASET)**

In [28]:
def normalizar_y_deduplicar(df):
    nombres_limpios = []
    for c in df.columns:
        limpio = c.strip().lower().replace(' ', '_').replace('/', '_').replace('.', '_')
        limpio = re.sub(r'[^a-z0-9_]', '', limpio)
        nombres_limpios.append(limpio)

    contador = Counter()
    nombres_finales = []
    for nombre in nombres_limpios:
        contador[nombre] += 1
        if contador[nombre] == 1:
            nombres_finales.append(nombre)
        else:
            nombres_finales.append(f"{nombre}_dup{contador[nombre]}")

    return df.toDF(*nombres_finales)

df_crudo = normalizar_y_deduplicar(df_crudo)
df_ml    = normalizar_y_deduplicar(df_ml)

print("Columnas df_crudo:", df_crudo.columns)

Columnas df_crudo: ['flow_id', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'protocol', 'timestamp', 'flow_duration', 'total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytes_s', 'flow_packets_s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packets_s', 'bwd_packets_s', 'min_packet_length', 'max_packet_length', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', 'fin_flag_count', 'syn_flag_count', 'rst

**DIAGNOSTICO**

In [29]:
def diagnostico_spark(df, nombre):
    print(f"=== {nombre} ===")
    df.printSchema()

    total = df.count()
    print(f"Filas: {total:,}")

    print("\nNulos por columna (solo columnas con > 0):")
    nulos = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
    ]).collect()[0].asDict()
    nulos_con_valor = {k: v for k, v in nulos.items() if v > 0}
    for col, cantidad in sorted(nulos_con_valor.items(), key=lambda x: -x[1]):
        print(f"  {col}: {cantidad:,}")

    duplicados = total - df.dropDuplicates().count()
    print(f"\nFilas duplicadas: {duplicados:,}")
    print("="*50)

diagnostico_spark(df_crudo, "df_crudo")
diagnostico_spark(df_ml, "df_ml")

=== df_crudo ===
root
 |-- flow_id: string (nullable = true)
 |-- source_ip: string (nullable = true)
 |-- source_port: double (nullable = true)
 |-- destination_ip: string (nullable = true)
 |-- destination_port: double (nullable = true)
 |-- protocol: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- flow_duration: double (nullable = true)
 |-- total_fwd_packets: double (nullable = true)
 |-- total_backward_packets: double (nullable = true)
 |-- total_length_of_fwd_packets: double (nullable = true)
 |-- total_length_of_bwd_packets: double (nullable = true)
 |-- fwd_packet_length_max: double (nullable = true)
 |-- fwd_packet_length_min: double (nullable = true)
 |-- fwd_packet_length_mean: double (nullable = true)
 |-- fwd_packet_length_std: double (nullable = true)
 |-- bwd_packet_length_max: double (nullable = true)
 |-- bwd_packet_length_min: double (nullable = true)
 |-- bwd_packet_length_mean: double (nullable = true)
 |-- bwd_packet_length_std: double (nulla

**VERIFICAR SI LAS COLUMNAS DUPLICADAS SON EXACTAMENTE IGUALES**

In [30]:
print("Solo fwd_header_length:", df_crudo.select("fwd_header_length").distinct().count())
print("Combinación de ambas:", df_crudo.select("fwd_header_length", "fwd_header_length_1").distinct().count())

Solo fwd_header_length: 3772
Combinación de ambas: 3772


**ELIMINAR DUPLICADA**

In [7]:
df_crudo = df_crudo.drop("fwd_header_length_1")
df_ml    = df_ml.drop("fwd_header_length_1")

print("Columnas restantes en df_crudo:", len(df_crudo.columns))
print("Columnas restantes en df_ml:", len(df_ml.columns))

Columnas restantes en df_crudo: 84
Columnas restantes en df_ml: 78


**NORMALIZACION DE VALORES**

>Tomando en cuenta la cantidad de campos que existen como double se debe revisar los que guardan otro tipo de informacion, como id, fechas y descripciones


**NORMALIZACION DE VALORES DE TEXTO**

In [8]:
def normalizar_valores_texto_spark(df):
    columnas_texto = [c for c, tipo in df.dtypes if tipo == 'string']
    print("Columnas de texto detectadas:", columnas_texto)
    for c in columnas_texto:
        df = df.withColumn(c, F.trim(F.col(c)))
    return df

df_crudo = normalizar_valores_texto_spark(df_crudo)
df_ml    = normalizar_valores_texto_spark(df_ml)

Columnas de texto detectadas: ['flow_id', 'source_ip', 'destination_ip', 'timestamp', 'label']
Columnas de texto detectadas: ['label']


**REVISAR VALORES UNICOS**

In [9]:
print(f"=== label en df_crudo ===")

# Cuántos nulos hay en label
nulos_label = df_crudo.filter(F.col("label").isNull()).count()
print(f"Filas con label nulo: {nulos_label:,}")

# Valores no nulos, ordenados
valores_crudo = [r[0] for r in df_crudo.select("label").distinct().collect() if r[0] is not None]
print(sorted(valores_crudo))

=== label en df_crudo ===
Filas con label nulo: 288,602
['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack \x96 Brute Force', 'Web Attack \x96 Sql Injection', 'Web Attack \x96 XSS']


In [10]:
# ¿Las filas sin label vienen de un archivo/fuente específica?
df_crudo.filter(F.col("label").isNull()).groupBy("source_file").count().show() if "source_file" in df_crudo.columns else print("No hay columna source_file para rastrear el origen")

No hay columna source_file para rastrear el origen


In [11]:
print("Nulos en label (df_ml):", df_ml.filter(F.col("label").isNull()).count())

Nulos en label (df_ml): 0


In [12]:
df_crudo = df_crudo.withColumn("label", F.regexp_replace(F.col("label"), "\x96", "-"))
df_ml    = df_ml.withColumn("label", F.regexp_replace(F.col("label"), "\x96", "-"))

# Verificar
valores_corregidos = [r[0] for r in df_crudo.select("label").distinct().collect() if r[0] is not None]
print(sorted(valores_corregidos))

['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


In [13]:
#Se verifica el contenido del resto de campos string
df_crudo.select("flow_id", "source_ip", "destination_ip").show(5)

+--------------------+-------------+--------------+
|             flow_id|    source_ip|destination_ip|
+--------------------+-------------+--------------+
|192.168.10.5-8.25...|8.254.250.126|  192.168.10.5|
|192.168.10.5-8.25...|8.254.250.126|  192.168.10.5|
|192.168.10.5-8.25...|8.254.250.126|  192.168.10.5|
|192.168.10.5-8.25...|8.254.250.126|  192.168.10.5|
|192.168.10.14-8.2...|8.253.185.121| 192.168.10.14|
+--------------------+-------------+--------------+
only showing top 5 rows



**NORMALIZACION DE VALORES DE FECHA**

In [14]:
#Se toma la columna relacionada a fechas para saber en que formato existen en el dataframe
df_crudo.select("timestamp").show(5)

+-------------------+
|          timestamp|
+-------------------+
|03/07/2017 08:55:58|
|03/07/2017 08:55:58|
|03/07/2017 08:55:58|
|03/07/2017 08:55:58|
|03/07/2017 08:56:22|
+-------------------+
only showing top 5 rows



In [15]:
# Convertir el texto a formato de fecha/hora indicando el formato

#Internamente el df posee valores distintos, por medio de coalesce se tomara el formato por defecto 
df_crudo = df_crudo.withColumn(
    "timestamp",
    F.expr("coalesce(try_to_timestamp(timestamp, 'dd/MM/yyyy HH:mm:ss'), try_to_timestamp(timestamp, 'd/M/yyyy H:mm'))")
)

# Verificaciónó
df_crudo.select("timestamp").printSchema()

root
 |-- timestamp: timestamp (nullable = true)



**NORMALIZACION DE VALORES ABSOLUTOS**

In [16]:
#Se verifica y toma en cuenta los campos que deberia ser valores enteros por se contadores, de modo que modificando su tipo ayuda a la carga 
#de procesos

df_crudo.select(
    "source_port", "destination_port", "protocol", 
    "total_fwd_packets", "total_backward_packets",
    "fin_flag_count", "syn_flag_count", "rst_flag_count", 
    "psh_flag_count", "ack_flag_count", "urg_flag_count", 
    "cwe_flag_count", "ece_flag_count"
).show(3, truncate=False)

#segun el analisis deberia arrojar datos como .00

+-----------+----------------+--------+-----------------+----------------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+
|source_port|destination_port|protocol|total_fwd_packets|total_backward_packets|fin_flag_count|syn_flag_count|rst_flag_count|psh_flag_count|ack_flag_count|urg_flag_count|cwe_flag_count|ece_flag_count|
+-----------+----------------+--------+-----------------+----------------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+
|80.0       |49188.0         |6.0     |2.0              |0.0                   |0.0           |0.0           |0.0           |0.0           |1.0           |1.0           |0.0           |0.0           |
|80.0       |49188.0         |6.0     |2.0              |0.0                   |0.0           |0.0           |0.0           |0.0           |1.0           |1.0           |0.0           |0.0        

In [17]:
#Se agargan todas las columnas en una lista para recorrer el proceso en un ciclo
cambiotipo_columnas = [
    "source_port", "destination_port", "protocol", 
    "total_fwd_packets", "total_backward_packets",
    "fin_flag_count", "syn_flag_count", "rst_flag_count", 
    "psh_flag_count", "ack_flag_count", "urg_flag_count", 
    "cwe_flag_count", "ece_flag_count"
]
#se cambia el tipo de dato por entero
for c in cambiotipo_columnas:
    df_crudo = df_crudo.withColumn(c, F.col(c).cast(IntegerType()))

# Comprobación rapida con algunas columnas
df_crudo.select("source_port", "protocol", "fin_flag_count").printSchema()

root
 |-- source_port: integer (nullable = true)
 |-- protocol: integer (nullable = true)
 |-- fin_flag_count: integer (nullable = true)



**ELIMINAR FILAS NULAS DE DF_CRUDO**

In [18]:
antes = df_crudo.count()
df_crudo = df_crudo.filter(F.col("label").isNotNull())
print(f"df_crudo: {antes:,} -> {df_crudo.count():,} filas ({antes - df_crudo.count():,} filas sin etiqueta eliminadas)")

df_crudo: 3,119,345 -> 2,830,743 filas (288,602 filas sin etiqueta eliminadas)


**DUPLICADOS DE FILA**

In [19]:

antes = df_crudo.count()
df_crudo = df_crudo.dropDuplicates()
print(f"df_crudo: {antes:,} -> {df_crudo.count():,} filas")

df_crudo: 2,830,743 -> 2,830,540 filas


In [20]:
# Distribución de label ANTES de guardar
df_ml.groupBy("label").count().orderBy(F.desc("count")).show(20, truncate=False)

+----------------------------+-------+
|label                       |count  |
+----------------------------+-------+
|BENIGN                      |2273097|
|DoS Hulk                    |231073 |
|PortScan                    |158930 |
|DDoS                        |128027 |
|DoS GoldenEye               |10293  |
|FTP-Patator                 |7938   |
|SSH-Patator                 |5897   |
|DoS slowloris               |5796   |
|DoS Slowhttptest            |5499   |
|Bot                         |1966   |
|Web Attack ï¿½ Brute Force  |1507   |
|Web Attack ï¿½ XSS          |652    |
|Infiltration                |36     |
|Web Attack ï¿½ Sql Injection|21     |
|Heartbleed                  |11     |
+----------------------------+-------+



**GUARDAR**

In [21]:
#Ruta de guardado y declaracion de archivo parquet
ruta_crudo_final = os.path.join(ANALITICOS_PARQUET, "dataset_crudo_limpio_final.parquet")

#Guardado del dataframe de Spark en formato parquet (via pandas, evitando el mecanismo de escritura de Hadoop)
df_pandas_crudo = df_crudo.toPandas()
df_pandas_crudo.to_parquet(ruta_crudo_final, index=False, engine="pyarrow")

print("Guardado en:", ruta_crudo_final)

Guardado en: ..\Datos\Analiticos\Parquet\dataset_crudo_limpio_final.parquet


**REVERIFICACION DE NULOS**

In [22]:
nulos_crudo = df_crudo.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_crudo.columns
]).collect()[0].asDict()
nulos_crudo_con_valor = {k: v for k, v in nulos_crudo.items() if v > 0}
print("\nNulos restantes en df_crudo:")
for col, cantidad in sorted(nulos_crudo_con_valor.items(), key=lambda x: -x[1]):
    print(f"  {col}: {cantidad:,}")

Nulos restantes en df_ml:
  flow_bytes_s: 1,358

Nulos restantes en df_crudo:
  flow_bytes_s: 1,357


In [23]:
#CERRAR SESION DE SPARK

spark.stop()
print("Sesion de espark finalizada")

Sesion de espark finalizada
